# 01 — The Potential Outcomes Framework
**References:** Rubin (1974) · Holland (1986) "Statistics and Causal Inference" (*JASA*) · Imbens & Rubin (2015) Ch. 1–2 · Hernán & Robins (2020) Ch. 1 · Stanford STATS 361 Lecture 1

## Narrative thread
```
Counterfactuals -> Potential outcomes -> Fundamental problem -> Estimands (ATE/ATT) -> Selection bias -> Randomization as solution
```

## From association to causation

For each unit $i$ and a binary treatment $W_i \in \{0, 1\}$, define **two potential outcomes**:

- $Y_i(1)$ — the outcome unit $i$ *would have* if treated
- $Y_i(0)$ — the outcome unit $i$ *would have* if not treated

The **individual treatment effect** is $\tau_i = Y_i(1) - Y_i(0)$.

We only ever observe one of the two (the *observed* outcome):
$$Y_i^{obs} = W_i\, Y_i(1) + (1 - W_i)\, Y_i(0)$$

> **The Fundamental Problem of Causal Inference** (Holland 1986): it is impossible to
> observe both $Y_i(1)$ and $Y_i(0)$ on the same unit. Causal inference is a
> **missing data problem** — half of the science table is always missing.

## SUTVA — the assumption behind the notation

Writing $Y_i(w)$ at all requires the **Stable Unit Treatment Value Assumption** (Rubin 1980):

1. **No interference:** unit $i$'s outcome does not depend on other units' treatments
   (violated by network spillovers, vaccines, marketplace experiments)
2. **No hidden versions of treatment:** the treatment is well-defined — "aspirin" is one thing

## Estimands

| Estimand | Definition | Question it answers |
|---|---|---|
| ATE | $\tau = E[Y(1) - Y(0)]$ | Effect of treating everyone vs. no one |
| ATT | $\tau_t = E[Y(1) - Y(0) \mid W = 1]$ | Effect on those who actually got treated |
| ATC | $\tau_c = E[Y(1) - Y(0) \mid W = 0]$ | Effect on those who did not |
| CATE | $\tau(x) = E[Y(1) - Y(0) \mid X = x]$ | Effect for units with covariates $x$ (notebook 11) |


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)

In [ ]:
# ── The science table: what we can and cannot see ────────────────────────
# Simulate a small population where BOTH potential outcomes are known to us
# (because we are the simulator — in real life this table is never complete).

n = 10
baseline  = np.random.normal(50, 10, n)          # health without treatment
y0 = baseline
y1 = baseline + np.random.normal(8, 4, n)        # heterogeneous individual effects
tau_i = y1 - y0

# Suppose sicker people (low y0) seek treatment: selection on the baseline
p_treat = 1 / (1 + np.exp((y0 - 50) / 5))
w = np.random.binomial(1, p_treat)
y_obs = np.where(w == 1, y1, y0)

science = pd.DataFrame({
    'W': w,
    'Y(0)': y0.round(1), 'Y(1)': y1.round(1), 'tau_i': tau_i.round(1),
    'Y_obs': y_obs.round(1),
})
# What the analyst actually sees: the counterfactual column is missing
observed = science.copy()
observed.loc[observed['W'] == 1, 'Y(0)'] = np.nan
observed.loc[observed['W'] == 0, 'Y(1)'] = np.nan
observed['tau_i'] = np.nan

print('The COMPLETE science table (only the simulator sees this):')
print(science.to_string(index_names=False))
print(f'\nTrue ATE = mean(tau_i) = {tau_i.mean():.2f}')
print('\nWhat the analyst sees (fundamental problem — half is missing):')
print(observed.to_string(index_names=False))

## Selection bias: why the naive comparison fails

The naive difference in observed means decomposes as (Angrist & Pischke 2009, §2.2):

$$\underbrace{E[Y^{obs} \mid W=1] - E[Y^{obs} \mid W=0]}_{\text{naive difference}}
= \underbrace{E[Y(1) - Y(0) \mid W=1]}_{\text{ATT}}
+ \underbrace{E[Y(0) \mid W=1] - E[Y(0) \mid W=0]}_{\text{selection bias}}$$

Selection bias is the difference in *untreated* potential outcomes between the groups —
"the treated would have been different anyway." If sicker people seek treatment,
$E[Y(0) \mid W=1] < E[Y(0) \mid W=0]$ and the naive difference *understates* the benefit
(or even flips its sign).

## Randomization solves it

If $W$ is assigned at random, then $W \perp (Y(0), Y(1))$, so
$E[Y(0) \mid W=1] = E[Y(0) \mid W=0]$: the selection-bias term is exactly zero and

$$E[Y^{obs} \mid W=1] - E[Y^{obs} \mid W=0] = E[Y(1)] - E[Y(0)] = \text{ATE}$$

Randomization makes the missing counterfactuals **missing completely at random**.


In [ ]:
# ── Selection bias decomposition, verified by simulation ──────────────────
n = 200_000
y0 = np.random.normal(50, 10, n)
y1 = y0 + np.random.normal(8, 4, n)      # true ATE = 8

# Scenario A: self-selection (sicker -> more likely treated)
w_sel = np.random.binomial(1, 1 / (1 + np.exp((y0 - 50) / 5)))
# Scenario B: randomized assignment
w_rct = np.random.binomial(1, 0.5, n)

for name, w in [('Self-selected', w_sel), ('Randomized', w_rct)]:
    y_obs  = np.where(w == 1, y1, y0)
    naive  = y_obs[w == 1].mean() - y_obs[w == 0].mean()
    att    = (y1 - y0)[w == 1].mean()
    selbias = y0[w == 1].mean() - y0[w == 0].mean()
    print(f'{name}:')
    print(f'  naive diff = {naive:6.2f}   ATT = {att:5.2f}   selection bias = {selbias:6.2f}')
    print(f'  check: ATT + selection bias = {att + selbias:6.2f}\n')

print(f'True ATE = {(y1 - y0).mean():.2f}')
print('Self-selection makes the treatment look HARMFUL even though it helps everyone.')

In [ ]:
# ── Visualize: the counterfactual is what identification is about ────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)

for ax, w, title in [(axes[0], w_sel, 'Self-selected: groups differ at baseline'),
                     (axes[1], w_rct, 'Randomized: groups comparable')]:
    idx = np.random.choice(n, 3000, replace=False)
    ax.hist(y0[idx][w[idx] == 1], bins=40, alpha=0.55, label='$Y(0)$ of treated (unseen)', color='#d62728')
    ax.hist(y0[idx][w[idx] == 0], bins=40, alpha=0.55, label='$Y(0)$ of control (seen)', color='#1f77b4')
    ax.set_title(title)
    ax.set_xlabel('untreated potential outcome $Y(0)$')
    ax.legend(fontsize=8)

axes[0].set_ylabel('count')
plt.tight_layout()
plt.show()

## Identification vs. estimation

Two distinct steps, kept separate throughout this course:

1. **Identification** — can the causal estimand be written as a function of the
   *distribution of observable data*, given the assumptions? (A question about the world,
   answered with $n = \infty$.)
2. **Estimation** — given a finite sample, how do we estimate that function well?
   (A statistics question: bias, variance, confidence intervals.)

No amount of data fixes an identification failure. The naive comparison above is a perfect
estimator — of the wrong quantity.

## Key takeaways

| Concept | Statement |
|---|---|
| Potential outcomes | Each unit has $Y(1), Y(0)$; the causal effect is their difference |
| Fundamental problem | Only one potential outcome is ever observed per unit |
| SUTVA | No interference between units; one version of treatment |
| ATE / ATT | Population vs. treated-subpopulation average effects |
| Selection bias | naive diff = ATT + $\{E[Y(0)\mid W{=}1] - E[Y(0)\mid W{=}0]\}$ |
| Randomization | Makes $W \perp (Y(0), Y(1))$, killing selection bias by design |
| Identification ≠ estimation | Assumptions link estimands to data; statistics then estimates |
